In [2]:
# Import the removed metals dataset – removed manually:
import pandas as pd
import numpy as np
import itertools
from rdkit import Chem
from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, rdMolDescriptors
import networkx as nx

import seaborn as sns
import matplotlib.pyplot as plt
from feature_selection import structural_similarity, mw_diff

full_dataset = pd.read_csv("../datasets/B3DB_classification_annotated.csv")
smiles = full_dataset['SMILES']

# Check validity by generating fp using rdkit.
invalid = []
fpgen = AllChem.GetRDKitFPGenerator()
for smile_id, smile in enumerate(smiles): 
    try: 
        mol = Chem.MolFromSmiles(smile)
        fpgen.GetFingerprint(mol)
    except: 
        invalid.append(smile_id)

# Remove invalid structures. 
all_valid_dataset = full_dataset.drop(invalid, axis=0)
all_valid_dataset = all_valid_dataset.reset_index()
smiles_valid = all_valid_dataset['SMILES']
logBB_valid = all_valid_dataset['logBB']
bbb_class_valid = all_valid_dataset['Class']

print('Dataset size after removing metals and unstandardizable smiles structures: ', len(all_valid_dataset))

# Calculate structural similarity and MW to identify stereoisomers.
sim_vals, _, _ = structural_similarity(smiles_valid)
sim_matrix_valid = np.array(sim_vals).reshape((len(smiles_valid), len(smiles_valid)))
mw_diff_matrix = np.array(mw_diff(smiles_valid)).reshape((len(smiles_valid), len(smiles_valid)))

sim_inds, mw_inds= np.column_stack(np.where(sim_matrix_valid == 1)),  np.column_stack(np.where(mw_diff_matrix == 0))
sim_inds, mw_inds = np.array([np.array([x[0], x[1]]) for x in sim_inds if x[0]<x[1]]), np.array([np.array([x[0], x[1]]) for x in mw_inds if x[0]<x[1]])
drop_inds = [ list(el) for el in sim_inds if el in mw_inds]

# Find clusters of stereoisomers and select the one with the lowest index (most likely to have a logBB value)
G = nx.Graph()
G.add_edges_from(drop_inds)
clusters = list(nx.connected_components(G))
drop_ind_clusters = [sorted(list(c))[1:] for c in clusters]
drop_ind_clusters_flat = [item for sublist in drop_ind_clusters for item in sublist]

# Remove all redundant stereoisomers
all_valid_dataset_no_str = all_valid_dataset.drop(drop_ind_clusters_flat, axis=0)
all_valid_dataset_no_str = all_valid_dataset_no_str.reset_index()


Dataset size after removing metals and unstandardizable smiles structures:  7807


In [3]:

output_file = '../datasets/not_druglike_b3db.csv'

base = all_valid_dataset_no_str


drug_like_mask = (
    base["MW"].le(800)
    & base["nHBDon"].le(6)
    & base["nHBAcc"].le(11)
    & base["SLogP"].le(5)
    & base["SLogP"].gt(0)
    & base["TopoPSA"].lt(180)
)

all_valid_dataset_drug_like = base.loc[drug_like_mask].copy()
all_valid_dataset_not_drug_like = base.loc[~drug_like_mask].copy()

all_valid_dataset_not_drug_like.to_csv(output_file,index=False,)

print("Cleaned valid dataset:", len(base))
print("Drug-like subset:", len(all_valid_dataset_drug_like))
print("Exact complement:", len(all_valid_dataset_not_drug_like))

Cleaned valid dataset: 4016
Drug-like subset: 3156
Exact complement: 860


#### generate the 3D sdf file of the output file from the previous step and then proceed: 

DataWarrior input file: '../datasets/not_druglike_b3db.csv'
DataWarrior output file: '../datasets/not_druglike_b3db.sdf'

In [4]:
from rdkit.Chem.PandasTools import LoadSDF
from rdkit import RDLogger
import pandas as pd
from rdkit import Chem 
from rdkit.Chem import AllChem, MACCSkeys 
import numpy as np

In [12]:
RDLogger.DisableLog('rdApp.warning')   

# Load the unlabelled dataset
data_warrior_3D_file = '../datasets/not_druglike_b3db.sdf'

df = LoadSDF(data_warrior_3D_file, smilesName='SMILES')
SMILES = df['SMILES']
bbb_class = df['Class']
logBB = df['logBB']
supervised_label_df = df[['SMILES', 'logBB', 'Class']]

In [ ]:
RDLogger.DisableLog('rdApp.warning')   

# ECFP
fpgen = AllChem.GetMorganGenerator(radius=2)
ecfp_fingerprints = []
for smile in SMILES:
    mol = Chem.MolFromSmiles(smile)
    fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2).ToBitString()
    ecfp_fingerprints.append(list(fp))

ecfp_col_names = [f"ECFP_{i}" for i in range(0,2048)]
ecfp_df = pd.DataFrame(ecfp_fingerprints, columns=ecfp_col_names)

# Mordred
mordred_output_file = "../datasets/b3db_undruglike_mordred_labels_with_id.csv"

! python -m mordred -3 {data_warrior_3D_file} -o {mordred_output_file}
mordred_df = pd.read_csv(mordred_output_file)


# MACCS
def get_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol).ToList())

col_names = [f"MACCS_{i}" for i in range(0,167)]
maccs_fps = []
for smile in SMILES: 
    maccs_fps.append(get_maccs_fingerprint(smile))

maccs_fp_df = pd.DataFrame(data=maccs_fps, columns=col_names)

# all_labelled_df = pd.concat([SMILES, bbb_class, ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
# all_labelled_df.to_csv('../datasets/b3db_not_drug_like_labelled.csv')

# Save dataset:
all_labelled_df = pd.concat([SMILES, ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
inverted_classes = (1-np.array([int(i)for i in np.array(all_labelled_df['Class'])]))*0.5
all_labelled_df['Class'] = [int(ilabel) for ilabel in inverted_classes]
all_labelled_df.to_csv('../../datasets/not_druglike_b3db_labelled.csv')

[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[18:03:14] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

KeyError: 'Class'

In [13]:
all_labelled_df = pd.concat([SMILES, logBB, bbb_class, ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
inverted_classes = (1-np.array([int(i)for i in np.array(all_labelled_df['Class'])]))*0.5
all_labelled_df['Class'] = [int(ilabel) for ilabel in inverted_classes]
all_labelled_df.to_csv('../datasets/not_druglike_b3db_labelled.csv')

In [ ]:
# bbb_class = df['Class']
# bbb_class=(bbb_class).astype(int)
# bbb_class= (bbb_class*(-1)+1)/2
# all_labelled_df = pd.concat([SMILES, bbb_class.astype(int), ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
# all_labelled_df.to_csv('../datasets/all_valid_dataset_breaks_SLogP_gt_5_labelled.csv')

# bbb_class